# Exact Entry Forecast Experiments

Ce notebook reprend le script `exact_entry_forecast_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Prevoit l'instant exact d'entree physique avec metriques causales.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Exact physical-entry forecast experiment with multi-horizon and multi-bin targets.
- Run par defaut : `runs/exp_105_exact_entry_forecast`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "exact_entry_forecast_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import time_to_entry_experiments as tte
from ml_pipeline import ROOT, safe_auc, write_json
from sequence_experiments import set_seed
from time_to_entry_repeated_split_experiments import (
    create_run_dir,
    device_from_arg,
    load_raw_sequence_data,
    load_sequence_data,
    normalize_for_split,
    repeated_split_maps,
    summarize,
    with_base_model_name,
)


EXACT_TIME_BINS = [
    ("no_entry_or_near_miss", None),
    ("post_entry", (-math.inf, 0.0)),
    ("pre_00_02", (0.0, 0.2)),
    ("pre_02_03", (0.2, 0.3)),
    ("pre_03_05", (0.3, 0.5)),
    ("pre_05_10", (0.5, 1.0)),
    ("pre_gt_10", (1.0, math.inf)),
]

EXACT_SURVIVAL_HORIZONS = [0.2, 0.3, 0.5, 1.0]


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `configure_exact_targets`

Cette cellule definit `configure_exact_targets`. Elle prepare une partie du script.

In [ ]:
def configure_exact_targets():
    tte.TIME_BINS = EXACT_TIME_BINS
    tte.SURVIVAL_HORIZONS = EXACT_SURVIVAL_HORIZONS


## Fonction `make_multibin_targets_exact`

Cette cellule definit `make_multibin_targets_exact`. Elle prepare une partie du script.

In [ ]:
def make_multibin_targets_exact(meta):
    y = np.zeros(len(meta), dtype=np.int64)
    tte_vals = meta["time_to_target_numeric"].to_numpy(dtype=np.float32)
    is_entry = meta["has_physical_entry"].to_numpy(dtype=bool)

    y[is_entry & (tte_vals <= 0.0)] = 1
    y[is_entry & (tte_vals > 0.0) & (tte_vals <= 0.2)] = 2
    y[is_entry & (tte_vals > 0.2) & (tte_vals <= 0.3)] = 3
    y[is_entry & (tte_vals > 0.3) & (tte_vals <= 0.5)] = 4
    y[is_entry & (tte_vals > 0.5) & (tte_vals <= 1.0)] = 5
    y[is_entry & (tte_vals > 1.0)] = 6

    weights = np.ones(len(meta), dtype=np.float32)
    weights[meta["is_hard_negative"].to_numpy(dtype=bool)] = 1.50
    weights[is_entry & (tte_vals <= 0.0)] = 0.40
    return y, weights


## Fonction `multibin_score_frame_exact`

Cette cellule definit `multibin_score_frame_exact`. Elle prepare une partie du script.

In [ ]:
def multibin_score_frame_exact(meta, probs, model_name, train_time_s, inference_s):
    pred = meta.copy()
    pred["objective"] = "multibin_exact"
    pred["model_name"] = model_name
    pred["train_time_s"] = float(train_time_s)
    pred["inference_ms_per_window"] = float(inference_s * 1000.0 / max(len(pred), 1))
    for idx, (name, _) in enumerate(EXACT_TIME_BINS):
        pred[f"p_{name}"] = probs[:, idx]
    pred["score_pre_any"] = probs[:, 2:].sum(axis=1)
    pred["score_pre_ge02"] = probs[:, 3:].sum(axis=1)
    pred["score_pre_ge03"] = probs[:, 4:].sum(axis=1)
    pred["score_pre_00_03"] = probs[:, 2:4].sum(axis=1)
    pred["score_pre_00_05"] = probs[:, 2:5].sum(axis=1)
    pred["score_pre_03_10"] = probs[:, 4:6].sum(axis=1)
    return pred


## Fonction `survival_score_frame_exact`

Cette cellule definit `survival_score_frame_exact`. Elle prepare une partie du script.

In [ ]:
def survival_score_frame_exact(meta, probs, model_name, train_time_s, inference_s):
    pred = meta.copy()
    pred["objective"] = "survival_exact"
    pred["model_name"] = model_name
    pred["train_time_s"] = float(train_time_s)
    pred["inference_ms_per_window"] = float(inference_s * 1000.0 / max(len(pred), 1))
    for idx, horizon in enumerate(EXACT_SURVIVAL_HORIZONS):
        pred[f"p_entry_by_{horizon:.1f}s"] = probs[:, idx]
    h = {horizon: probs[:, idx] for idx, horizon in enumerate(EXACT_SURVIVAL_HORIZONS)}
    pred["score_by_02"] = h[0.2]
    pred["score_by_03"] = h[0.3]
    pred["score_by_05"] = h[0.5]
    pred["score_by_10"] = h[1.0]
    pred["score_future_02_10"] = np.maximum(h[1.0] - h[0.2], 0.0)
    pred["score_future_03_10"] = np.maximum(h[1.0] - h[0.3], 0.0)
    return pred


## Fonction `score_columns_for_objective_exact`

Cette cellule definit `score_columns_for_objective_exact`. Elle prepare une partie du script.

In [ ]:
def score_columns_for_objective_exact(objective):
    if objective == "multibin":
        return [
            "score_pre_any",
            "score_pre_ge02",
            "score_pre_ge03",
            "score_pre_00_03",
            "score_pre_00_05",
            "score_pre_03_10",
        ]
    return [
        "score_by_02",
        "score_by_03",
        "score_by_05",
        "score_by_10",
        "score_future_02_10",
        "score_future_03_10",
    ]


## Fonction `validation_score_exact`

Cette cellule definit `validation_score_exact`. Elle prepare une partie du script.

In [ ]:
def validation_score_exact(objective, logits, meta, val_idx):
    val_meta = meta.iloc[val_idx]
    target = tte.future_entry_target(val_meta, max_horizon=0.5, min_horizon=0.0)
    if objective == "multibin":
        probs = torch.softmax(torch.from_numpy(logits[val_idx]), dim=1).numpy()
        score = probs[:, 2:5].sum(axis=1)
    else:
        probs = 1.0 / (1.0 + np.exp(-logits[val_idx]))
        score = probs[:, EXACT_SURVIVAL_HORIZONS.index(0.5)]
    return float(safe_auc(tte.average_precision_score, target, score) or 0.0)


## Fonction `evaluate_prediction_frame_exact`

Cette cellule definit `evaluate_prediction_frame_exact`. Elle prepare une partie du script.

In [ ]:
def evaluate_prediction_frame_exact(pred, score_cols, thresholds, persistence_values, repeat_seed):
    rows = []
    for score_col in score_cols:
        for threshold in thresholds:
            for persistence in persistence_values:
                for split in ["val", "test"]:
                    row = tte.evaluate_causal(pred, score_col, threshold, split, persistence)
                    row["objective"] = str(pred["objective"].iloc[0])
                    row["model_name"] = str(pred["model_name"].iloc[0])
                    row["inference_ms_per_window"] = float(pred["inference_ms_per_window"].iloc[0])
                    row["train_time_s"] = float(pred["train_time_s"].iloc[0])
                    row["repeat_seed"] = int(repeat_seed)
                    rows.append(row)
    return rows


## Fonction `reliability_rows`

Cette cellule definit `reliability_rows`. Elle prepare une partie du script.

In [ ]:
def reliability_rows(pred, score_col, split, repeat_seed, thresholds=(0.5, 0.7, 0.8, 0.9, 0.95)):
    sdf = pred[pred["split"].eq(split)].copy()
    tte_vals = pd.to_numeric(sdf["time_to_target_numeric"], errors="coerce")
    has_entry = sdf["has_physical_entry"].astype(bool)
    rows = []
    for threshold in thresholds:
        mask = pd.to_numeric(sdf[score_col], errors="coerce").ge(float(threshold))
        n = int(mask.sum())
        if n == 0:
            continue
        rows.append(
            {
                "repeat_seed": int(repeat_seed),
                "objective": str(pred["objective"].iloc[0]),
                "model_name": str(pred["model_name"].iloc[0]),
                "score_col": score_col,
                "split": split,
                "threshold": float(threshold),
                "n_windows": n,
                "entry_le_02_rate": float((has_entry & tte_vals.gt(0.0) & tte_vals.le(0.2) & mask).sum() / n),
                "entry_le_03_rate": float((has_entry & tte_vals.gt(0.0) & tte_vals.le(0.3) & mask).sum() / n),
                "entry_le_05_rate": float((has_entry & tte_vals.gt(0.0) & tte_vals.le(0.5) & mask).sum() / n),
                "entry_le_10_rate": float((has_entry & tte_vals.gt(0.0) & tte_vals.le(1.0) & mask).sum() / n),
                "mean_time_to_entry_s": float(tte_vals[mask & has_entry & tte_vals.gt(0.0)].mean()) if (mask & has_entry & tte_vals.gt(0.0)).any() else np.nan,
                "median_time_to_entry_s": float(tte_vals[mask & has_entry & tte_vals.gt(0.0)].median()) if (mask & has_entry & tte_vals.gt(0.0)).any() else np.nan,
            }
        )
    return rows


## Fonction `summarize_reliability`

Cette cellule definit `summarize_reliability`. Elle prepare une partie du script.

In [ ]:
def summarize_reliability(df):
    metric_cols = [
        "entry_le_02_rate",
        "entry_le_03_rate",
        "entry_le_05_rate",
        "entry_le_10_rate",
        "mean_time_to_entry_s",
        "median_time_to_entry_s",
    ]
    rows = []
    for keys, group in df.groupby(["objective", "base_model_name", "score_col", "split", "threshold"]):
        row = dict(zip(["objective", "base_model_name", "score_col", "split", "threshold"], keys))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        row["n_windows_total"] = int(group["n_windows"].sum())
        for col in metric_cols:
            vals = pd.to_numeric(group[col], errors="coerce")
            row[f"{col}_mean"] = float(vals.mean())
            row[f"{col}_std"] = float(vals.std(ddof=0))
        rows.append(row)
    return pd.DataFrame(rows)


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, selected_test, reliability_summary, best_test):
    lines = [
        "# Exact Entry Forecast Experiment",
        "",
        "This run treats the physical entry timestamp as the hard target and trains two causal families:",
        "",
        "- multi-horizon probability model for entry within `0.2 / 0.3 / 0.5 / 1.0s`",
        "- fine-grained multi-bin time-to-entry classifier around the exact entry instant",
        "",
        "## Validation-Selected Test Policy",
        "",
        "| field | value |",
        "|---|---:|",
    ]
    for key in [
        "objective",
        "base_model_name",
        "score_col",
        "threshold",
        "persistence_windows",
        "pre_entry_recall_mean",
        "early_03_recall_mean",
        "early_05_recall_mean",
        "event_precision_mean",
        "false_alarms_per_min_mean",
        "median_early_warning_s_mean",
    ]:
        value = selected_test.get(key, "")
        if isinstance(value, float):
            value = f"{value:.3f}"
        lines.append(f"| {key} | {value} |")

    lines.extend(
        [
            "",
            "## Best Test Rows",
            "",
            "| model | score | thr | persist | >=0.3s | >=0.5s | early/danger | precision | FA/min |",
            "|---|---|---:|---:|---:|---:|---:|---:|---:|",
        ]
    )
    for _, row in best_test.head(12).iterrows():
        lines.append(
            f"| {row['base_model_name']} | {row['score_col']} | {row['threshold']:.2f} | {int(row['persistence_windows'])} | "
            f"{row['early_03_recall_mean']:.3f} | {row['early_05_recall_mean']:.3f} | "
            f"{int(row['early_05_detected_total'])}/{int(row['danger_total'])} | {row['event_precision_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} |"
        )

    lines.extend(
        [
            "",
            "## Reliability Read",
            "",
            "These rows answer the practical question: when the model score is high, how often does entry really happen soon after?",
            "",
            "| model | score | split | thr | windows | P(entry<=0.2s) | P(entry<=0.3s) | P(entry<=0.5s) | P(entry<=1.0s) | mean tte |",
            "|---|---|---|---:|---:|---:|---:|---:|---:|---:|",
        ]
    )
    rel = reliability_summary[reliability_summary["split"].eq("test")].sort_values(
        ["threshold", "entry_le_03_rate_mean", "entry_le_05_rate_mean"],
        ascending=[False, False, False],
    )
    for _, row in rel.head(24).iterrows():
        lines.append(
            f"| {row['base_model_name']} | {row['score_col']} | {row['split']} | {row['threshold']:.2f} | {int(row['n_windows_total'])} | "
            f"{row['entry_le_02_rate_mean']:.3f} | {row['entry_le_03_rate_mean']:.3f} | {row['entry_le_05_rate_mean']:.3f} | {row['entry_le_10_rate_mean']:.3f} | "
            f"{row['mean_time_to_entry_s_mean']:.3f} |"
        )

    lines.extend(
        [
            "",
            "## Artifacts",
            "",
            f"- Causal metrics: `{run_dir / 'metrics' / 'exact_entry_causal_summary.csv'}`",
            f"- Reliability summary: `{run_dir / 'metrics' / 'exact_entry_reliability_summary.csv'}`",
            f"- Reliability raw rows: `{run_dir / 'metrics' / 'exact_entry_reliability_raw.csv'}`",
            f"- Training summary: `{run_dir / 'metrics' / 'exact_entry_training_summary.csv'}`",
        ]
    )
    (run_dir / "exact_entry_forecast_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


## Fonction `parse_model_specs`

Cette cellule definit `parse_model_specs`. Elle prepare une partie du script.

In [ ]:
def parse_model_specs(values):
    specs = []
    for value in values:
        objective, kind = value.split(":", 1)
        specs.append((objective, kind))
    return specs


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    configure_exact_targets()
    set_seed(args.seed)
    device = device_from_arg(args.device)
    run_dir = create_run_dir(args.run_name)
    X_raw = load_raw_sequence_data(args.sequence_run)
    _, base_meta, sequence_run, base_run = load_sequence_data(args.sequence_run, args.base_run)
    split_maps = repeated_split_maps(args.old_score_run)
    thresholds = [round(float(x), 2) for x in np.arange(args.threshold_min, args.threshold_max + 1e-9, args.threshold_step)]
    persistence_values = [int(x) for x in args.persistence_windows]

    multibin_y, multibin_weights = make_multibin_targets_exact(base_meta)
    survival_y, survival_weights = tte.make_survival_targets(base_meta)

    write_json(
        run_dir / "config.json",
        {
            "sequence_run": str(resolve(args.sequence_run)),
            "base_run": str(resolve(args.base_run)),
            "old_score_run": str(resolve(args.old_score_run)),
            "time_bins": [name for name, _ in EXACT_TIME_BINS],
            "survival_horizons": EXACT_SURVIVAL_HORIZONS,
            "model_specs": args.model_specs,
            "thresholds": thresholds,
            "persistence_values": persistence_values,
            "device": str(device),
        },
    )

    all_metric_rows = []
    all_reliability_rows = []
    all_ap_rows = []
    all_history = []
    train_rows = []
    split_rows = []

    original_validation_score = tte.validation_score
    tte.validation_score = validation_score_exact
    try:
        for repeat_seed, split_map in sorted(split_maps.items()):
            meta = base_meta.copy()
            meta["split"] = meta["video_id"].map(split_map)
            X, mean, std = normalize_for_split(X_raw, meta)
            meta.to_csv(run_dir / "features" / f"split_seed_{repeat_seed}.csv", index=False)
            np.savez_compressed(run_dir / "features" / f"normalizer_seed_{repeat_seed}.npz", mean=mean, std=std)
            split_rows.append(
                {
                    "repeat_seed": int(repeat_seed),
                    **{f"{split}_videos": int(meta[meta["split"].eq(split)]["video_id"].nunique()) for split in ["train", "val", "test"]},
                    **{f"{split}_danger_videos": int(meta[(meta["split"].eq(split)) & (meta["is_danger_clip"].eq(1))]["video_id"].nunique()) for split in ["train", "val", "test"]},
                }
            )
            for objective, kind in args.model_specs:
                name = f"seed{repeat_seed}_{objective}_{kind}_exact_entry"
                y = multibin_y if objective == "multibin" else survival_y
                sample_weights = multibin_weights if objective == "multibin" else survival_weights
                print(f"training {name} on {device}")
                model, history, train_time_s, model_size_bytes, best = tte.train_one(objective, kind, name, X, y, sample_weights, meta, run_dir, args, device)
                for row in history:
                    row["repeat_seed"] = int(repeat_seed)
                all_history.extend(history)

                start = time.perf_counter()
                logits = tte.predict_logits(model, X, args.batch_size, device)
                inference_s = time.perf_counter() - start
                if objective == "multibin":
                    probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
                    pred = multibin_score_frame_exact(meta, probs, name, train_time_s, inference_s)
                else:
                    probs = 1.0 / (1.0 + np.exp(-logits))
                    pred = survival_score_frame_exact(meta, probs, name, train_time_s, inference_s)
                pred["repeat_seed"] = int(repeat_seed)
                pred.to_csv(run_dir / "features" / f"predictions_{name}.csv", index=False)

                score_cols = score_columns_for_objective_exact(objective)
                all_metric_rows.extend(evaluate_prediction_frame_exact(pred, score_cols, thresholds, persistence_values, repeat_seed))
                for split in ["val", "test"]:
                    for score_col in score_cols:
                        all_reliability_rows.extend(reliability_rows(pred, score_col, split, repeat_seed))
                for row in tte.ap_auc_rows(pred, score_cols):
                    row["repeat_seed"] = int(repeat_seed)
                    all_ap_rows.append(row)
                train_rows.append(
                    {
                        "repeat_seed": int(repeat_seed),
                        "objective": objective,
                        "kind": kind,
                        "model_name": name,
                        "best_epoch": int(best["epoch"]),
                        "best_val_ap_future_0_5s": float(best["score"]),
                        "train_time_s": float(train_time_s),
                        "model_size_bytes": int(model_size_bytes),
                        "inference_ms_per_window": float(inference_s * 1000.0 / max(len(meta), 1)),
                    }
                )
    finally:
        tte.validation_score = original_validation_score

    metrics = pd.DataFrame(all_metric_rows)
    metrics = tte.add_selection_score(metrics)
    metrics = with_base_model_name(metrics)
    ap_metrics = pd.DataFrame(all_ap_rows)
    reliability = pd.DataFrame(all_reliability_rows)
    reliability["base_model_name"] = reliability["model_name"].astype(str).str.replace(r"^seed\d+_", "", regex=True)
    reliability_summary = summarize_reliability(reliability)
    history_df = pd.DataFrame(all_history)
    train_df = pd.DataFrame(train_rows)
    split_df = pd.DataFrame(split_rows)

    summary = summarize(
        metrics,
        ["objective", "base_model_name", "score_col", "threshold", "persistence_windows", "split"],
        ["selection_score", "pre_entry_recall", "early_03_recall", "early_05_recall", "event_precision", "false_alarms_per_min", "median_early_warning_s"],
    )

    metrics.to_csv(run_dir / "metrics" / "exact_entry_causal_metrics.csv", index=False)
    summary.to_csv(run_dir / "metrics" / "exact_entry_causal_summary.csv", index=False)
    reliability.to_csv(run_dir / "metrics" / "exact_entry_reliability_raw.csv", index=False)
    reliability_summary.to_csv(run_dir / "metrics" / "exact_entry_reliability_summary.csv", index=False)
    ap_metrics.to_csv(run_dir / "metrics" / "exact_entry_ap_metrics.csv", index=False)
    history_df.to_csv(run_dir / "metrics" / "exact_entry_training_history.csv", index=False)
    train_df.to_csv(run_dir / "metrics" / "exact_entry_training_summary.csv", index=False)
    split_df.to_csv(run_dir / "metrics" / "exact_entry_split_audit.csv", index=False)

    val_rows = summary[summary["split"].eq("val")].sort_values("selection_score_mean", ascending=False)
    selected_val = val_rows.iloc[0]
    selected_test = summary[
        summary["objective"].eq(selected_val["objective"])
        & summary["base_model_name"].eq(selected_val["base_model_name"])
        & summary["score_col"].eq(selected_val["score_col"])
        & summary["threshold"].eq(selected_val["threshold"])
        & summary["persistence_windows"].eq(selected_val["persistence_windows"])
        & summary["split"].eq("test")
    ].iloc[0].to_dict()
    best_test = summary[summary["split"].eq("test")].sort_values("selection_score_mean", ascending=False)
    write_summary(run_dir, selected_test, reliability_summary, best_test)
    print(run_dir)
    print(run_dir / "exact_entry_forecast_summary.md")


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Exact physical-entry forecast experiment with multi-horizon and multi-bin targets.")
    parser.add_argument("--sequence-run", default="runs/exp_072_physical_entry_seq30_focal")
    parser.add_argument("--base-run", default="runs/exp_070_physical_entry_baseline")
    parser.add_argument("--old-score-run", default="runs/exp_088_physical_entry_final_score_full")
    parser.add_argument("--run-name", default="exp_105_exact_entry_forecast")
    parser.add_argument("--model-spec", action="append", default=["survival:tcn", "survival:gru", "multibin:tcn", "multibin:gru"])
    parser.add_argument("--epochs", type=int, default=26)
    parser.add_argument("--patience", type=int, default=6)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.04)
    parser.add_argument("--focal-gamma", type=float, default=1.5)
    parser.add_argument("--threshold-min", type=float, default=0.05)
    parser.add_argument("--threshold-max", type=float, default=0.95)
    parser.add_argument("--threshold-step", type=float, default=0.05)
    parser.add_argument("--persistence-windows", nargs="+", type=int, default=[1, 2])
    parser.add_argument("--device", default="auto")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    args.model_specs = parse_model_specs(args.model_spec)
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_105_exact_entry_forecast_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["exact_entry_forecast_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
